## 少量实例的提示词模版的使用
### FewShotPromptTemplate :与PromptTemplate一起使用
### FewShotChatMessagePromptTemplate:与ChatPromptTemplate一起使用
### Example selectors（实例选择器）

In [2]:

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

from hello_qwen import response

example_prompt=PromptTemplate(
    template="input:{input},output:{output}",
)

examples=[
    {"input":"北京天气怎么样？","output":"北京"},
    {"input":"青岛天气怎么样？","output":"青岛"},
    {"input":"天津天气怎么样？","output":"天津"}
]

few_shot_template=FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="input:{input},output:",
    input_variables=["input"]
)
few_shot_template.invoke({"input":"合肥会下雨吗？"})

StringPromptValue(text='input:北京天气怎么样？,output:北京\n\ninput:青岛天气怎么样？,output:青岛\n\ninput:天津天气怎么样？,output:天津\n\ninput:合肥会下雨吗？,output:')

调用大模型后：

In [8]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os
import dotenv
dotenv.load_dotenv()
llm=ChatOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
    model="deepseek-chat",
)
example_prompt=PromptTemplate(
    template="原句:{input},改写:{output}",
)

examples=[
    {"input":"北京天气怎么样？","output":"北京"},
    {"input":"青岛天气怎么样？","output":"青岛"},
    {"input":"天津天气怎么样？","output":"天津"}
]

few_shot_template=FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="原句:{输入},改写：",
    input_variables=["输入"]
)
prompt=few_shot_template.invoke({"输入":"合肥天气怎么样？"})
response=llm.invoke(prompt)
print(response.content)

合肥


## FewShotChatMessagePromptTemplate

In [3]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

examples=[
    {"input":"2@2","output":"4"},
    {"input":"2@3","output":"8"},
    {"input":"2@4","output":"16"}
]
example_prompt=ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}"),
    ]
)

few_shot_template=FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)
print(few_shot_template)
print(type(few_shot_template))
print(few_shot_template.invoke({}))

examples=[{'input': '2@2', 'output': '4'}, {'input': '2@3', 'output': '8'}, {'input': '2@4', 'output': '16'}] input_variables=[] input_types={} partial_variables={} example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], input_types={}, partial_variables={}, template='{output}'), additional_kwargs={})])
<class 'langchain_core.prompts.few_shot.FewShotChatMessagePromptTemplate'>
messages=[HumanMessage(content='2@2', additional_kwargs={}, response_metadata={}), AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='2@3', additional_kwargs={}, response_metadata={}), AIMessage(content='8', additional_kwargs={}, response_metadata=

In [12]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

examples=[
    {"input":"2@2","output":"4"},
    {"input":"2@3","output":"8"},
    {"input":"2@4","output":"16"}
]
example_prompt=ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}"),
    ]
)

few_shot_template=FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
)

final_prompt=ChatPromptTemplate(
    [
        ("system","你是一个数学天才"),
        few_shot_template,
        ("human","{input}"),
    ]
)
prompt=final_prompt.invoke({"input":"2@5"})
response=llm.invoke(prompt)
print(response.content)

32


In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_chroma import Chroma
import dotenv
import os

dotenv.load_dotenv()
# llm=ChatOpenAI(
#     api_key=os.getenv("OPENAI_API_KEY"),
#     base_url=os.getenv("OPENAI_BASE_URL"),
#     temperature=0,
#     model="gpt-3.5-turbo",
# )
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"]=os.getenv("OPENAI_BASE_URL")
#定义嵌入模型
embeddings_model=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

#定义实例组
examples=[
    {"question": "小丽最喜欢小林", "answer": "说明小丽对小林最有好感"},
    {"question": "小明最爱打篮球", "answer": "说明小明最喜欢的运动是篮球"},
    {"question": "小红最喜欢吃苹果", "answer": "说明小红最喜欢的水果是苹果"},
]

#定义实例选择器
example_selector=SemanticSimilarityExampleSelector.from_examples(
    #这是可供选择的实例列表
    examples,
    #这是用于生成嵌入的嵌入类，用于衡量语义相似性
    embeddings_model,
    #这是用于存储嵌入并进行相似性搜素的VectorStore类
    Chroma,
    #这是要生成的示例数量
    k=1,
)

#选择与输入最相似的示例
question = "小丽最爱小林"
selected_examples=example_selector.select_examples({"question":question})
print(f"与输入最相似的示例：{selected_examples}")


与输入最相似的示例：[{'question': '小丽最喜欢小林', 'answer': '说明小丽对小林最有好感'}]


#### 举例2：

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import FewShotPromptTemplate,PromptTemplate
from langchain_openai import OpenAIEmbeddings
#定义实例提示词模版
example_prompt=PromptTemplate.from_template(
    template="输入:{input},输出:{output}"
)

#创建一个实例提示词模版
examples=[
    {"input":"高兴","output":"悲伤"},
    {"input":"精力充沛","output":"无精打采"},
    {"input":"阳光","output":"阴暗"},
    {"input":"粗糙","output":"光滑"},
    {"input":"干燥","output":"潮湿"},
    {"input":"富裕","output":"贫穷"},
]

#定义嵌入模型
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small"
)

#创建语义相似性示例选择器
example_selector=SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    FAISS,
    k=2,
)

#定义小样本提示词模版
similar_prompt=FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个词组的反义词",
    suffix="输入：{word},输出：，",
    input_variables=["word"]
)

response=similar_prompt.invoke({"word":"忧郁"})
print(response.text)

给出每个词组的反义词

输入:高兴,输出:悲伤

输入:富裕,输出:贫穷

输入：忧郁,输出：，
